<font size="6" color='grey'><b>KI-Agenten. Verstehen. Anwenden. Gestalten.</b></font>

<font size="5" color='grey'><b>Projekt-Template B – Content-Team</b></font>

---

> **Vorlage für M23 Multi-Agent-Projekt**
> Suche relevante `# ⚠️ TODO`-Kommentare und passe sie an dein Thema an.

**Team-Struktur:**

| Rolle | Agent | Aufgabe |
|-------|-------|---------|
| 🎯 Koordination | **Supervisor** | Routing & Planung |
| ✍️ Erstellung | **Writer** | Texte verfassen |
| 🔍 Qualität | **Editor** | Texte prüfen & verbessern |

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
# ⚠️ TODO: Passe den Projektnamen an deinen Team-Namen an
os.environ["LANGSMITH_PROJECT"]  = "M23-Content-Team"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

## 1 | Übersicht & Lernziel
---

Dieses Template implementiert ein **Content-Team** bestehend aus:
- **Writer:** Verfasst Texte zu einem vorgegebenen Thema
- **Editor:** Prüft den Text und gibt konstruktives Feedback
- **Supervisor:** Koordiniert Writer → Editor → FINISH

**Typischer Workflow:**
```
START → Supervisor → Writer → Supervisor → Editor → Supervisor → FINISH
```

> Der Editor kann den Supervisor anweisen, den Writer nochmals zu schicken – das ist Schreib-Iteration in der Praxis.

In [ ]:
#@title
#@markdown <p><font size="4" color='green'>flowchart: Content-Team-Architektur</font></p>
diagram = '''
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nContent-Manager"]

    SUP -->|"writer"| W["✍️ Writer\nText erstellen"]
    SUP -->|"editor"| E["🔍 Editor\nText prüfen & verbessern"]
    SUP -->|"FINISH"| END([END])

    W -->|"AIMessage\n(name=Writer)"| SUP
    E -->|"AIMessage\n(name=Editor)"| SUP

    W --- WT1["📄 draft_text"]
    W --- WT2["🔢 count_words"]
    E --- ET1["✅ check_quality"]
    E --- ET2["📝 suggest_improvements"]

    style SUP fill:#FF9800,color:#fff
    style W   fill:#4CAF50,color:#fff
    style E   fill:#2196F3,color:#fff
    style WT1 fill:#E8F5E9
    style WT2 fill:#E8F5E9
    style ET1 fill:#E3F2FD
    style ET2 fill:#E3F2FD
'''
mermaid(diagram, width=780)

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent
from IPython.display import Image as IPImage

In [ ]:
# Worker-LLM: gpt-4o-mini (Inhalte generieren, schnell & günstig)
worker_llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

# Supervisor-LLM: o3 (Routing-Entscheidungen, starkes Reasoning)
# ⚠️ o3 akzeptiert KEINEN temperature-Parameter (API-Fehler)
from langchain_core.messages import SystemMessage as SM
import langchain
_base_supervisor = init_chat_model("openai:o3")
print("✅ LLMs initialisiert (worker: gpt-4o-mini | supervisor: o3)")

In [ ]:
# ── Writer Tools ───────────────────────────────────────────────────────────

@tool
def draft_text(thema: str, laenge: str = "mittel") -> str:
    """Hilft beim Strukturieren eines Textentwurfs.
    laenge: 'kurz' (100–150 W), 'mittel' (200–300 W), 'lang' (400+ W)
    """
    laengen_map = {"kurz": "100–150", "mittel": "200–300", "lang": "400+"}
    ziel = laengen_map.get(laenge, "200–300")
    return (
        f"Text-Vorlage für '{thema}' ({ziel} Wörter):\n"
        "- Einleitungssatz: Kernthese benennen\n"
        "- Hauptteil: 2–3 Argumente mit Beispielen\n"
        "- Schluss: Fazit und Handlungsempfehlung\n"
        f"Ziel: Klarer, prägnanter Text mit ca. {ziel} Wörtern."
    )

@tool
def count_words(text: str) -> str:
    """Zählt Wörter in einem Text – zur Längenkontrolle."""
    anzahl = len(text.split())
    if anzahl < 100:
        hinweis = "→ Text zu kurz, bitte ausbauen."
    elif anzahl > 400:
        hinweis = "→ Text sehr lang, ggf. kürzen."
    else:
        hinweis = "→ Länge passt."
    return f"Wortanzahl: {anzahl}. {hinweis}"

# ── Editor Tools ───────────────────────────────────────────────────────────

@tool
def check_quality(text: str) -> str:
    """Prüft einen Text auf häufige Qualitätsprobleme.
    Gibt eine Bewertung (1–10) und Verbesserungshinweise zurück.
    """
    punkte = []
    score  = 8  # Startwert
    if len(text.split()) < 50:
        punkte.append("⚠️ Text zu kurz (< 50 Wörter)")
        score -= 2
    if text.count("!") > 3:
        punkte.append("⚠️ Zu viele Ausrufezeichen")
        score -= 1
    if not any(text.endswith(x) for x in [".", "!", "?"]):
        punkte.append("⚠️ Kein sauberer Schlusssatz")
        score -= 1
    if not punkte:
        punkte.append("✅ Keine offensichtlichen Probleme gefunden")
    return f"Qualitätsscore: {score}/10\n" + "\n".join(punkte)

@tool
def suggest_improvements(aspekt: str) -> str:
    """Gibt konkrete Verbesserungsvorschläge für einen bestimmten Aspekt.
    aspekt: z.B. 'Einleitung', 'Argumentation', 'Schluss', 'Stil'
    """
    vorschlaege = {
        "einleitung": "Hook-Satz zuerst, dann These. Keine allgemeinen Aussagen.",
        "argumentation": "Jedes Argument: Behauptung → Begründung → Beispiel.",
        "schluss": "Kernaussage wiederholen, konkreten nächsten Schritt nennen.",
        "stil": "Aktive Formulierungen bevorzugen. Sätze max. 20 Wörter.",
    }
    key = aspekt.lower()
    for k, v in vorschlaege.items():
        if k in key:
            return f"Verbesserung '{aspekt}': {v}"
    return f"Allgemeiner Tipp für '{aspekt}': Klar, konkret, kurz."

print("✅ Tools definiert: draft_text, count_words, check_quality, suggest_improvements")

In [ ]:
# ── Worker Agents ──────────────────────────────────────────────────────────

# ⚠️ TODO: Passe die System-Prompts an dein Content-Thema an

writer_agent = create_react_agent(
    worker_llm,
    tools=[draft_text, count_words],
    prompt=SystemMessage(
        # ⚠️ TODO: Beschreibe Stil und Zielgruppe des Textes
        "Du bist ein erfahrener Texter. "
        "Nutze draft_text für die Struktur und count_words zur Längenkontrolle. "
        "Schreibe klar, prägnant und zielgruppengerecht. "
        "Ziel: ca. 200–300 Wörter. Antworte auf Deutsch."
    )
)

editor_agent = create_react_agent(
    worker_llm,
    tools=[check_quality, suggest_improvements],
    prompt=SystemMessage(
        # ⚠️ TODO: Passe Qualitätskriterien an dein Projekt an
        "Du bist ein kritischer Editor. "
        "Nutze check_quality zur Bewertung und suggest_improvements für Feedback. "
        "Gib konstruktives, konkretes Feedback. "
        "Wenn der Text gut ist (Score ≥ 7), sage explizit: 'Text genehmigt'. "
        "Antworte auf Deutsch."
    )
)

print("✅ Worker Agents erstellt:")
print("   writer_agent → [draft_text, count_words]")
print("   editor_agent → [check_quality, suggest_improvements]")

In [ ]:
# ── State Definition ───────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages:    Annotated[list, add_messages]  # Gesamte Nachrichten-Historie
    aufgabe:     str    # Original-Aufgabe (unveränderlich)
    naechster:   str    # Routing-Entscheidung des Supervisors
    iterationen: int    # Zähler gegen Endlosschleifen
    max_iter:    int    # Konfigurierbare Grenze (Standard: 6)

# ── Routing-Entscheidung mit Begründung ────────────────────────────────────
class SupervisorEntscheidung(BaseModel):
    """Strukturierte Routing-Entscheidung des Supervisors."""
    naechster: Literal["writer", "editor", "FINISH"] = Field(
        description="Nächster Agent oder FINISH wenn die Aufgabe vollständig ist."
    )
    begruendung: str = Field(
        description="Kurze Begründung für diese Entscheidung (1–2 Sätze)."
    )

supervisor_llm = (
    _base_supervisor
    .with_structured_output(SupervisorEntscheidung)
    .with_retry(stop_after_attempt=3)
)
print("✅ State und Supervisor-LLM konfiguriert")

In [ ]:
SUPERVISOR_PROMPT = (
    "Du bist Content-Manager eines Schreib-Teams.\n\n"
    "<Team>\n"
    "- writer: Schreibt den Text (IMMER zuerst)\n"
    "- editor: Prüft Qualität und gibt Feedback (NACH writer)\n"
    "</Team>\n\n"
    # ⚠️ TODO: Passe den Workflow an – z.B. max. Überarbeitungsrunden
    "<Workflow>\n"
    "Standard: writer → editor → FINISH\n"
    "Bei Editor-Feedback 'nicht genehmigt': writer → editor → FINISH\n"
    "Maximal 1 Überarbeitungsrunde (dann FINISH erzwingen).\n"
    "Wie du die Historie liest:\n"
    "  name=Writer → writer war aktiv\n"
    "  name=Editor → editor war aktiv\n"
    "</Workflow>\n\n"
    "<Rules>\n"
    "1. writer IMMER zuerst.\n"
    "2. editor nach writer – mindestens einmal.\n"
    "3. FINISH wenn editor 'genehmigt' sagt ODER nach max. 2 Writer-Runden.\n"
    "4. Nie mehr als 2x writer aufrufen.\n"
    "</Rules>"
)
print("✅ Supervisor-Prompt konfiguriert")

In [ ]:
# ── Supervisor Node ────────────────────────────────────────────────────────
def supervisor_node(state: AgentState) -> dict:
    """Liest Situation, entscheidet wer als Nächstes arbeitet."""
    # Iterations-Schutz: verhindert Endlosschleifen
    if state["iterationen"] >= state["max_iter"]:
        mprint(f"⚠️ **Iterations-Limit ({state['max_iter']}) erreicht – FINISH erzwungen.**")
        return {"naechster": "FINISH"}

    entscheidung = supervisor_llm.invoke([
        SystemMessage(SUPERVISOR_PROMPT),
        HumanMessage(f"Aufgabe: {state['aufgabe']}"),
        *state["messages"],
    ])
    mprint(f"**👔 Supervisor → `{entscheidung.naechster}`** – _{entscheidung.begruendung}_")
    return {"naechster": entscheidung.naechster}


# ── Worker Node Factory ────────────────────────────────────────────────────
def make_worker_node(agent, name: str):
    """Erzeugt einen Worker-Node mit Fehlerbehandlung und Kontext-Übergabe."""
    def worker_node(state: AgentState) -> dict:
        mprint(f"\n**🤖 {name}** arbeitet...")
        # Aufgabe + bisherige Ergebnisse als Kontext übergeben
        kontext = f"Aufgabe: {state['aufgabe']}"
        if state["messages"]:
            bisherige = "\n".join(
                f"{getattr(m, 'name', 'User')}: {m.content[:300]}"
                for m in state["messages"]
            )
            kontext += f"\n\nBisherige Arbeit des Teams:\n{bisherige}"

        try:
            result = agent.invoke(
                {"messages": [HumanMessage(kontext)]},
                config={"recursion_limit": 15}
            )
            antwort = result["messages"][-1].content
        except Exception as e:
            # ✅ MVP-Anforderung: Mind. ein Fehlerpfad abgefangen
            antwort = f"❌ Fehler: {e} – bitte alternativen Weg wählen."
            mprint(f"  ⚠️ {name} fehlgeschlagen: {e}")

        return {
            "messages":    [AIMessage(content=antwort, name=name)],
            "iterationen": state["iterationen"] + 1,
        }
    worker_node.__name__ = f"{name}_node"
    return worker_node


# ── Router ─────────────────────────────────────────────────────────────────
def supervisor_router(state: AgentState) -> str:
    naechster = state.get("naechster", "FINISH")
    return "__end__" if naechster == "FINISH" else naechster

print("✅ Supervisor-Node, Worker-Nodes und Router definiert")

In [ ]:
# ── Graph aufbauen ────────────────────────────────────────────────────────
builder = StateGraph(AgentState)

# Nodes
builder.add_node("supervisor", supervisor_node)
builder.add_node("writer", make_worker_node(writer_agent, "Writer"))
builder.add_node("editor", make_worker_node(editor_agent, "Editor"))

# Einstiegspunkt
builder.add_edge(START, "supervisor")

# Bedingtes Routing
builder.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "writer": "writer",
        "editor": "editor",
        "__end__": END,
    }
)

# Zurück zum Supervisor nach jedem Worker
builder.add_edge("writer",  "supervisor")
builder.add_edge("editor",  "supervisor")

graph = builder.compile()
print("✅ Graph kompiliert")

# Visualisierung
try:
    display(IPImage(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"(Visualisierung nicht verfügbar: {e})")

In [ ]:
# ── Testlauf ──────────────────────────────────────────────────────────────
# ⚠️ TODO: Passe die Testaufgabe an dein Team-Thema an
testaufgabe = "Schreibe einen kurzen Erklärtext (ca. 200 Wörter) über den Nutzen von KI-Agenten im Arbeitsalltag."

result = graph.invoke(
    {
        "messages":    [],
        "aufgabe":     testaufgabe,
        "naechster":   "",
        "iterationen": 0,
        "max_iter":    6,      # ✅ MVP-Schutz: max. 6 Worker-Calls
    },
    config={
        "run_name": "M23-Content-Team-Test",
        "tags":     ["m23-content"],
        "recursion_limit": 40,  # Äußere Grenze für den gesamten Graph
    }
)

mprint("## 🎯 Ergebnis")
mprint(f"**Aufgabe:** {testaufgabe}\n")
mprint(f"**Iterationen:** {result['iterationen']} von max. {result['max_iter']}\n---")
for msg in result["messages"]:
    agent = getattr(msg, "name", None) or "System"
    mprint(f"**[{agent}]**\n{msg.content}\n---")

In [ ]:
#@title 📊 LangSmith Trace{ display-mode: "form" }
import time; time.sleep(2)
show_trace("M23-Content-Team", limit=2, show_steps=True)

## ✅ MVP-Checkliste (vor Abgabe prüfen)

Dein Projekt ist **fertig**, wenn alle Punkte erfüllt sind:

| Kriterium | Geprüft? |
|-----------|----------|
| Supervisor routet zu **mindestens 2 Workern** | ☐ |
| Worker geben **sinnvolle Antworten** | ☐ |
| Mind. **ein Fehlerpfad** ist abgefangen (bereits im Template ✅) | ☐ |
| **End-to-End-Flow** funktioniert (1 Durchlauf ohne Fehler) | ☐ |
| **LangSmith-Trace** zeigt den Ablauf | ☐ |

**Nicht erforderlich:**
- ❌ Perfekte Ausgabe
- ❌ Schöne UI
- ❌ Mehr als 2 Worker

## A | Deine Aufgabe – Content-Team anpassen

---

### Was du bauen sollst

Du nimmst dieses Template und entwickelst daraus ein **konkretes Content-System** für einen selbst gewählten Inhalt und eine bestimmte Zielgruppe. Am Ende des Tages präsentierst du ein System, das Texte schreibt, sie durch einen Editor prüft und bei Bedarf überarbeitet.

---

### Schritt-für-Schritt-Anleitung

#### Schritt 1 – Content-Typ & Zielgruppe wählen (5 Min)

Einige sich im Team auf einen konkreten Anwendungsfall:

| Content-Typ | Zielgruppe | Beispiel-Aufgabe |
|------------|-----------|-----------------|
| 📰 Blogartikel | Einsteiger | „Erkläre KI-Agenten für technisch interessierte Laien" |
| 📧 Newsletter | Abonnenten | „Fasse die KI-Neuigkeiten der Woche in 200 Wörtern zusammen" |
| 📋 Produkttext | Kunden | „Beschreibe unser neues Feature in 150 Wörtern" |
| 📚 Lehrmaterial | Kursteilnehmer | „Erkläre LangGraph für Python-Entwickler mit 2 Jahren Erfahrung" |

Tragt euren Anwendungsfall ein: **Unser Content-Typ: ___________________________**

---

#### Schritt 2 – Writer-Tools anpassen (10 Min)

Öffne die **Tools**-Zelle und passe `draft_text` an deinen Content-Typ an:

```python
@tool
def draft_text(thema: str, laenge: str = "mittel") -> str:
    """..."""
    # ⚠️ TODO: Passe die Struktur an deinen Content-Typ an
    # Blogartikel:  Hook → These → 3 Argumente → CTA
    # Newsletter:   Lead → Hauptmeldung → Kurzmeldungen → Ausblick
    # Produkttext:  Problem → Lösung → Nutzen → Handlungsaufforderung
    return f"Vorlage für '{thema}' ({laenge}): ..."
```

**Tipp:** Du kannst `count_words` auch mit einer Warnung ergänzen, wenn das Wortlimit
für deinen Content-Typ über- oder unterschritten wird.

---

#### Schritt 3 – Editor-Qualitätskriterien definieren (10 Min)

Öffne `check_quality` und lege **eigene Qualitätskriterien** fest:

```python
@tool
def check_quality(text: str) -> str:
    """..."""
    # ⚠️ TODO: Welche Qualität ist für euren Content-Typ wichtig?
    # Blogartikel: Headline, Lesbarkeit, Keyword
    # Newsletter:  Länge max. 200 W, kein Fachjargon, konkreter CTA
    # Lehrmaterial: Beispiele vorhanden, Konzept erklärt, kein Copy-Paste
    punkte = []
    score  = 8
    # Deine Prüfungen hier:
    if ...:
        punkte.append("⚠️ ...")
        score -= 1
    return f"Score: {score}/10\n" + "\n".join(punkte)
```

---

#### Schritt 4 – System-Prompts & Supervisor-Logik anpassen (10 Min)

**Writer-Prompt:** Beschreibe Ton, Stil und Zielgruppe:
```python
"Du bist Texter für [ZIELGRUPPE]. Schreibe [TON: informell/formell/sachlich] ..."
```

**Editor-Prompt:** Welche Kriterien sind entscheidend?
```python
"Du bist Editor für [CONTENT-TYP]. Bewerte besonders: [KRITERIUM 1], [KRITERIUM 2] ..."
```

**Supervisor-Prompt:** Wie viele Überarbeitungsrunden sind sinnvoll?
Ändere den Supervisor-Prompt so, dass er maximal **1 Überarbeitungsrunde** zulässt
(Standard ist bereits konfiguriert – prüfe ob es zu deinem Use Case passt).

---

#### Schritt 5 – Testen & LangSmith prüfen (10 Min)

1. Passe `testaufgabe` in der Testlauf-Zelle auf euren Content-Typ an
2. Führe alle Zellen von oben nach unten aus
3. Beobachte in LangSmith: Hat der Editor den Text akzeptiert oder abgelehnt?
4. Hake die MVP-Checkliste ab

---

### Mini-Demo am Ende (3 Min pro Team)

Zeigt euren Mitstreitern:
1. **Eine Beispiel-Anfrage** → Welchen Text liefert das System?
2. **Editor-Feedback** → Was hat der Editor konkret bemängelt oder gelobt?
3. **LangSmith-Trace** → Wie viele Runden hat es gebraucht?

---

### Bonus-Aufgaben (wenn das MVP läuft)

**Bonus 1 – Style-Guide-Tool**
Erstelle ein `apply_style_guide`-Tool, das sicherstellt, dass der Text
bestimmte Formulierungen verwendet (oder vermeidet):

```python
@tool
def apply_style_guide(text: str) -> str:
    """Prüft Einhaltung des Style Guides (verbotene Wörter, Pflichtbegriffe)."""
    verboten  = ["eigentlich", "quasi", "irgendwie"]
    probleme  = [w for w in verboten if w in text.lower()]
    return f"Style-Check: {len(probleme)} Verstöße: {probleme}" if probleme else "Style-Check: OK"
```

**Bonus 2 – Versionierung**
Speichere jede Writer-Version mit Zeitstempel, damit der Editor verschiedene
Versionen vergleichen kann (→ ähnlich wie Git für KI-generierten Content).

**Bonus 3 – Human in the Loop**
Baue einen menschlichen Genehmigungsschritt ein: Bevor der finale Text
als „genehmigt" gilt, muss ein Mensch ihn freigeben (→ *M17 Human-in-the-Loop*).

**Bonus 4 – Mehrsprachigkeit**
Füge einen `translator_agent` als dritten Worker hinzu, der den genehmigten
Text in eine zweite Sprache übersetzt.
